In [ ]:
"""
run_event_regressions.py
------------------------
The three signed-event regressions on abnormal returns.

Pipeline (all components unit-tested on synthetic + real event windows):
  1. Load AR (rolling-beta output) and merge carbon intensity.
  2. Build the SIGNED event-window treatment from channel_signs:
       - ±1 trading-day windows on the panel's own trading-day grid
       - exact-date events only (month-precision TNACs excluded from tight windows)
       - confirmed overlap-free (22 events -> 66 window-dates, 0 collisions)
  3. Run three specifications, each: AR ~ carbon_z + (sign x carbon_z) | isin + date
       HEADLINE   : all events with a risk sign (pv_ar_sign != 0)   -> 14 events
       CLEAN-RISK : pure-uncertainty subsample (events 2,4,8,16,17) -> 5 events  [the sharp test]
       PLACEBO    : invalidations (events 19,23,25), supply sign    -> 3 events

Sign convention: the treatment is pv_ar_sign (resolution +1 / introduction -1),
so a POSITIVE interaction coefficient = effect in the Pastor-Veronesi-predicted
direction (amplification). Magnitude is sign-only (the news count is parked in
robustness). Expect the headline to be diluted relative to clean-risk, since the
confounded events drag it toward zero -- headline-weak + clean-significant is the
key result, not a failure.

Clustering: two-way (isin + date) for headline and clean-risk. The placebo has
only 3 events with a constant sign, where two-way date-clustering is unreliable
(can be singular or anti-conservative), so it uses firm clustering and should be
read as a low-power check on the point estimate, not a precise p-value.
"""

import numpy as np
import pandas as pd
import pyfixest as pf
import warnings
warnings.filterwarnings("ignore", message=".*multicollinearity.*")   # carbon_z absorbed by firm FE (expected)

# ----------------------------- config -----------------------------
AR_PATH      = "/Users/andradaduta/Desktop/Master Thesis/Python Data/AR/stoxx600_AR_2013_2025_w48.parquet"     # rolling-beta output (primary window)
EVENTS_PATH  = "/Users/andradaduta/Desktop/Master Thesis/Python Data/Events/EU_MSR_Events_sign.xlsx"                    # the AUGMENTED file (has channel_signs)
CARBON_PATH  = "/Users/andradaduta/Desktop/Master Thesis/Python Data/emissions/emissions_revenue_with_isin_key_final.xlsx"        # <-- your carbon file (see Section A)
ID, DATE     = "isin", "date"
CARBON       = "carbon_intensity_z"
WINDOW       = 1                                        # ±1 trading day (set 2 for robustness)
EVENT_START  = "2013-01-01"


# ===================================================================# ===================================================================
# ===================================================================
# SECTION A - load AR, build carbon z-score, merge (lagged + ffilled)
# ===================================================================
ar = pd.read_parquet(AR_PATH)
ar[DATE] = pd.to_datetime(ar[DATE])
ar = ar.dropna(subset=["AR"]).copy()
ar[ID] = ar[ID].astype(str).str.strip()
print(f"[A] AR panel: {len(ar):,} firm-days | {ar[DATE].dt.year.min()}-{ar[DATE].dt.year.max()} | {ar[ID].nunique()} firms")

# ---- build carbon intensity from raw emissions / revenue ----
carbon = pd.read_excel(CARBON_PATH)
carbon = carbon.rename(columns={"ISIN": "isin",
                                "CO2 Equivalent Emissions Direct, Scope 1": "scope1",
                                "Date": "carbon_date"})
carbon["isin"] = carbon["isin"].astype(str).str.strip()
carbon["carbon_date"] = pd.to_datetime(carbon["carbon_date"])
carbon["year"] = carbon["carbon_date"].dt.year
carbon = carbon.dropna(subset=["scope1", "Revenue"])
carbon = carbon[carbon["Revenue"] > 0]
carbon["carbon_intensity"] = carbon["scope1"] / carbon["Revenue"]

# drop non-positive intensity BEFORE logging (a hard 0 is a non-report, and log(0) is undefined)
n_zero = int((carbon["carbon_intensity"] <= 0).sum())
carbon = carbon[carbon["carbon_intensity"] > 0].copy()
print(f"[A] carbon obs: {len(carbon):,} | dropped {n_zero} zero/neg intensity | {carbon['isin'].nunique()} firms")
print(carbon["carbon_intensity"].describe())

# one row per (isin, year) so the merge can't fan out
carbon = carbon.groupby(["isin", "year"], as_index=False)["carbon_intensity"].mean()

# log (tames the ~12,000x skew) then z-score WITHIN year
carbon["carbon_intensity_z"] = carbon.groupby("year")["carbon_intensity"].transform(
    lambda s: (np.log(s) - np.log(s).mean()) / np.log(s).std())
carbon = carbon.dropna(subset=["carbon_intensity_z"])
print(f"[A] carbon_z: {len(carbon):,} firm-years | mean {carbon['carbon_intensity_z'].mean():+.2f} "
      f"std {carbon['carbon_intensity_z'].std():.2f} (expect ~0 / ~1)")

# key-overlap check BEFORE merge (distinguishes a key mismatch from a value problem)
overlap = set(carbon["isin"]) & set(ar["isin"])
print(f"[A] ISIN overlap carbon∩AR: {len(overlap)} (carbon {carbon['isin'].nunique()}, AR {ar['isin'].nunique()})")
assert len(overlap) > 0, "no shared ISINs -- inspect both isin columns for format mismatch"

# ---- lagged + forward-filled merge via merge_asof ----
# FY{t} emissions are the operative measure during calendar {t+1}; 'backward' also
# carries the latest known score into trailing years (e.g. 2025, where FY emissions
# aren't published yet) -- defensible since carbon intensity is highly persistent.
carbon["available_from"] = pd.to_datetime((carbon["year"] + 1).astype(int).astype(str) + "-01-01")
carbon = carbon.sort_values("available_from")
ar = ar.sort_values(DATE)
ar = pd.merge_asof(ar, carbon[["isin", "available_from", "carbon_intensity_z"]],
                   left_on=DATE, right_on="available_from", by="isin", direction="backward")

before  = len(ar)
matched = int(ar[CARBON].notna().sum())
yrs     = sorted(ar.loc[ar[CARBON].notna(), DATE].dt.year.unique())
print(f"[A] matched {matched:,}/{before:,} firm-days ({matched/before:.1%}) | carbon years present: {yrs}")

ar = ar.dropna(subset=[CARBON]).reset_index(drop=True)
print(f"[A] FINAL panel: {len(ar):,} firm-days | {ar[ID].nunique()} firms | {ar[DATE].dt.year.min()}-{ar[DATE].dt.year.max()}")
assert len(ar) > 0, "no carbon matched -- check the ISIN overlap line above"


In [ ]:
"""
run_event_regressions_clean.py
------------------------------
Signed-event abnormal-return regressions, restructured around channel identification.

Runs AFTER Section A of your notebook (which defines `ar` with columns
[isin, date, AR, carbon_intensity_z] and the constants ID, DATE, CARBON,
EVENTS_PATH, EVENT_START). Sections B-D below replace the old B-D.

WHAT CHANGED vs the previous version
  1. Primary spec is CLEAN_RISK (supply-neutral), not the pooled "headline".
     The pooled all-signed regression is demoted to a named robustness check
     (POOLED_SIGNED) because it mixes events whose supply sign REINFORCES the
     risk sign (biases the coefficient away from zero) with events whose supply
     sign OPPOSES it (biases toward zero) -- so its coefficient conflates the
     risk and cash-flow channels and cannot identify either.
  2. New sharp cut CLEAN_RISK_INTRO: clean_risk restricted to risk-INTRODUCING
     events, whose sign (+regulatory risk) is unambiguous -- sidesteps the
     resolving-event sign-coding question. CLEAN_RISK_RESOLV isolates the
     resolving cell so you can see if it drives anything.
  3. The cash-flow channel is bounded separately: SUPPLY_PLACEBO (pure supply,
     pv=0) and REINFORCING (risk & supply agree, so a cash-flow effect, if any,
     is largest here).
  4. Every subset is DERIVED from analysis_group + event_type already in the
     workbook -- nothing hand-keyed, nothing can drift out of sync.
  5. Bug fix: the neutral TNAC (ev22) is dropped BEFORE windows are built, so it
     can no longer collide with ev21 in the pre-window and silently delete a real
     treatment date. Plus a treat_dates guard in run_spec so an excluded event can
     never remove an included event's date.

Sign convention: treatment = sign x carbon_z, so +beta = high-carbon AR moves in
the sign-predicted direction on event dates (amplification). Coefficients are
reported in BASIS POINTS per 1-SD of carbon_z (1 bp = 0.01%).
"""

import numpy as np
import pandas as pd
import pyfixest as pf
import warnings
warnings.filterwarnings("ignore", message=".*multicollinearity.*")  # carbon_z absorbed by firm FE (expected)


# ===================================================================
# SECTION B - load events, build the signed event windows
# ===================================================================
ev = pd.read_excel(EVENTS_PATH, sheet_name="events", parse_dates=["event_date"])
cs = pd.read_excel(EVENTS_PATH, sheet_name="channel_signs")
ev = ev.merge(cs[["event_id", "analysis_group", "pv_ar_sign", "supply_ar_sign"]], on="event_id")

# keep datable, in-sample events only (month-precision TNACs can't anchor tight windows)
ev = ev[(ev["event_date"] >= EVENT_START) & (ev["date_precision"] == "exact")].copy()

# Only events that are TREATMENT in some spec get a window:
#   - any event with a risk sign (pv_ar_sign != 0), OR
#   - a supply placebo (pv=0 but treatment for the supply channel).
# Neutral/informational events (pv=0, not a placebo) get NO window -- their dates fall
# through to the control pool. This removes the ev21/ev22 pre-window collision at its
# source: a no-signal TNAC can no longer sit on, and delete, a real treatment date.
used = ev[(ev["pv_ar_sign"] != 0) | (ev["analysis_group"] == "supply_placebo")].copy()

tdays = pd.Index(sorted(ar[DATE].unique()))   # the panel's own trading-day grid

def build_window(events, tdays, lo_off, hi_off, tag):
    recs = []
    for _, e in events.iterrows():
        pos = min(tdays.searchsorted(e["event_date"]), len(tdays) - 1)  # non-trading date -> next trading day
        lo, hi = max(0, pos + lo_off), min(len(tdays) - 1, pos + hi_off)
        for d in tdays[lo:hi + 1]:
            recs.append({DATE: d, "event_id": e["event_id"], "window": tag,
                         "analysis_group": e["analysis_group"], "event_type": e["event_type"],
                         "pv_ar_sign": e["pv_ar_sign"], "supply_ar_sign": e["supply_ar_sign"]})
    return pd.DataFrame(recs)

announce = build_window(used, tdays, -1, +1, "announce")   # +/-1 trading-day window
pre      = build_window(used, tdays, -5, -2, "pre")        # anticipation window

# overlap check on each window independently (should be 0 among treatment events now)
for tag, w in [("announce", announce), ("pre", pre)]:
    n_dup = int(w[DATE].duplicated().sum())
    print(f"[B] {tag:8s}: {used.shape[0]} treatment events -> {len(w):3d} window-dates | overlapping dates: {n_dup}")
    if n_dup:
        info = (w.loc[w[DATE].duplicated(keep=False)].sort_values(DATE)
                  .groupby(DATE).agg(events=("event_id", lambda s: sorted(s.tolist())),
                                     groups=("analysis_group", list)).reset_index())
        print(info.to_string(index=False))
    assert n_dup == 0, f"{tag}: treatment windows collide -- resolve before pooling signs"


# ===================================================================
# SECTION C - define the specification ladder + the regression helper
# ===================================================================
g  = used.set_index("event_id")                                            # event_id -> row
ag = used.groupby("analysis_group")["event_id"].apply(lambda s: sorted(s)).to_dict()

clean_risk        = set(ag.get("clean_risk", []))                                              # supply-neutral, signed
clean_risk_intro  = {i for i in clean_risk if g.loc[i, "event_type"] == "risk_introducing"}    # 4, 16  (sign -1)
clean_risk_resolv = {i for i in clean_risk if g.loc[i, "event_type"] == "risk_resolving"}      # 2,8,17 (sign +1)
reinforcing       = set(ag.get("reinforcing", []))                                             # risk & supply AGREE
supply_placebo    = set(ag.get("supply_placebo", []))                                          # pure supply, pv=0
pooled_signed     = set().union(*[set(ag.get(k, [])) for k in                                   # old "headline"
                                  ["clean_risk", "confounded", "reinforcing", "risk_ambiguous_supply"]])

print("\n[C] specification event sets (derived from analysis_group + event_type):")
for nm, s in [("clean_risk", clean_risk), ("clean_risk_intro", clean_risk_intro),
              ("clean_risk_resolv", clean_risk_resolv), ("reinforcing", reinforcing),
              ("supply_placebo", supply_placebo), ("pooled_signed", pooled_signed)]:
    print(f"    {nm:18s} n={len(s):2d}  events={sorted(s)}")


def run_spec(panel, win, event_ids, signcol, vcov):
    """Fit one specification. Controls = all non-event days; treatment = this spec's
    event days only. Every OTHER treatment event's days are removed from the panel so
    they can't sit in the residual -- EXCEPT any date this spec itself treats
    (treat_dates guard), so an excluded event can never delete an included one.
    Treatment = signcol x carbon_z."""
    sel = win[win["event_id"].isin(event_ids)]
    assert not sel[DATE].duplicated().any(), "duplicate treated date within a spec"
    treat_dates = set(sel[DATE])
    drop_dates  = set(win.loc[~win["event_id"].isin(event_ids), DATE]) - treat_dates
    df = panel[~panel[DATE].isin(drop_dates)].copy()
    df["sn"]  = df[DATE].map(sel.set_index(DATE)[signcol]).fillna(0)
    df["snx"] = df["sn"] * df[CARBON]
    df = df.dropna(subset=["AR", CARBON])
    m = pf.feols(f"AR ~ {CARBON} + snx | {ID} + {DATE}", data=df, vcov=vcov)
    return m, sel["event_id"].nunique(), len(df)


# ===================================================================
# SECTION D - run the ladder over both windows
# ===================================================================
TWOWAY = {"CRV1": f"{ID} + {DATE}"}   # firm + date clustering
FIRM   = {"CRV1": ID}                  # firm only -- use when <=3 same-sign events,
                                       # where date-clustering on so few treated dates
                                       # is unreliable. Read the point estimate + CI,
                                       # NOT the p-value, for the FIRM-clustered rows.

#   tier          label                event-id set        sign column       vcov
specs = [
    ("identify",   "CLEAN_RISK",       clean_risk,         "pv_ar_sign",     TWOWAY),  # PRIMARY: supply-neutral
    ("identify",   "CLEAN_RISK_INTRO", clean_risk_intro,   "pv_ar_sign",     FIRM),    # sharpest: + intro-only
    ("diagnostic", "CLEAN_RISK_RESOLV",clean_risk_resolv,  "pv_ar_sign",     FIRM),    # resolving-sign sensitive
    ("channel",    "SUPPLY_PLACEBO",   supply_placebo,     "supply_ar_sign", FIRM),    # pure supply, pv=0
    ("channel",    "REINFORCING",      reinforcing,        "pv_ar_sign",     FIRM),    # risk & supply agree
    ("robustness", "POOLED_SIGNED",    pooled_signed,      "pv_ar_sign",     TWOWAY),  # conflates channels
]

rows = []
for wlabel, w in [("ANNOUNCE [-1,+1]", announce), ("PRE-EVENT [-5,-2]", pre)]:
    for tier, label, ids, signcol, vcov in specs:
        if not ids:
            continue
        m, nev, n = run_spec(ar, w, ids, signcol, vcov)
        b, se = m.coef()["snx"], m.se()["snx"]
        t, p  = m.tstat()["snx"], m.pvalue()["snx"]
        lo, hi = m.confint().loc["snx"].tolist()
        rows.append({"window": wlabel, "tier": tier, "spec": label,
                     "sign": signcol.split("_")[0], "clust": "2way" if vcov is TWOWAY else "firm",
                     "events": nev, "N": n,
                     "beta_bp": b * 1e4, "se_bp": se * 1e4, "t": t, "p": p,
                     "ci_lo_bp": lo * 1e4, "ci_hi_bp": hi * 1e4})

summary = pd.DataFrame(rows)
pd.set_option("display.float_format", lambda x: f"{x:+.2f}")
print("\n" + "=" * 100)
print("SIGNED-EVENT AMPLIFICATION  (AR ~ carbon_z + sign x carbon_z | isin + date)   coef in BASIS POINTS / 1-SD")
print("  +beta = high-carbon AR moves in the sign-predicted direction on event dates   (1 bp = 0.01%)")
print("=" * 100)
print(summary.to_string(index=False))

print("""
READING THE LADDER
  CLEAN_RISK is the identifying spec: every event is supply-neutral (supply_ar_sign=0),
  so the interaction is not contaminated by the cash-flow channel. CLEAN_RISK_INTRO is the
  same cut restricted to risk-INTRODUCING events (sign unambiguous: +regulatory risk) -- the
  sharpest but lowest-power test (firm-clustered; read the estimate and CI, not the p-value).
  CLEAN_RISK_RESOLV isolates the resolving events, whose +1 coding is the one to stress-test;
  if it diverges from CLEAN_RISK_INTRO, the resolving sign-coding is doing work.

  SUPPLY_PLACEBO (pv=0, supply-signed) and REINFORCING (risk & supply agree, so pv == supply
  here) bound the cash-flow channel: a flat placebo AND a flat reinforcing cell say the supply
  channel isn't hiding under the risk label. A move in REINFORCING absent from CLEAN_RISK would
  point to supply, not risk.

  POOLED_SIGNED is the old all-signed regression. It mixes events whose supply sign REINFORCES
  the risk sign (bias away from zero) with events whose supply sign OPPOSES it (bias toward
  zero), so its coefficient conflates the two channels -- report as robustness/context only,
  never as identification of the risk channel.

ROBUSTNESS (re-run, not shown): 60-month-beta AR file; winsorised-excess AR; quartile carbon
  dummy; +/-2 windows; count-weighted treatment (pv_ar_sign x news_intensity). Note: AR padding
  (zero-return holiday rows) only inflates SEs in these first-moment specs -- de-pad only before
  any second-moment (variance) test.
""")

In [ ]:
#robustness +/- 2 windows 

In [ ]:
# ---------------------------------------------------------------
# ROBUSTNESS: energy-return x carbon control  -> Table 3 row
# Absorbs the one confound date FE can't reach: an energy-price move
# that hits high-carbon firms differentially on an event date.
# Run AFTER Cell 1 (needs: ar, announce, pre, clean_risk, ID/DATE/CARBON).
# ---------------------------------------------------------------
import pandas as pd, numpy as np, pyfixest as pf

# 1) Brent from FRED (public) -- NOT the EUA carbon price (that sits on the
#    causal path of your MSR events, so controlling for it is a bad control).
try:
    import pandas_datareader.data as web
    brent = web.DataReader("DCOILBRENTEU", "fred", "2012-01-01", "2025-12-31")
    brent = brent.rename(columns={"DCOILBRENTEU": "brent"}).reset_index().rename(columns={"DATE": "date"})
except Exception:
    url = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=DCOILBRENTEU"
    brent = pd.read_csv(url, na_values=".", parse_dates=["observation_date"])
    brent = brent.rename(columns={"observation_date": "date", "DCOILBRENTEU": "brent"})
brent = brent.dropna(subset=["brent"]).sort_values("date")

# 2) ffill the price onto YOUR panel's trading-day grid, then daily return.
#    ffill keeps any event date that falls on a Brent (US-holiday) gap, so no
#    treated date silently drops -- energy_ret is ~0 on such a day, which is benign.
pdx  = pd.Index(sorted(ar[DATE].unique()))
bser = brent.set_index("date")["brent"]
bser = bser.reindex(pdx.union(bser.index)).ffill()
energy = bser.reindex(pdx).pct_change().rename("energy_ret").rename_axis(DATE).reset_index()

# 3) merge + interaction. Do NOT add energy_ret as its own regressor: it's
#    constant across firms within a date, so your date FE absorbs the level.
ar_e = ar.merge(energy, on=DATE, how="left")
ar_e["energy_x_carbon"] = ar_e["energy_ret"] * ar_e[CARBON]

# 4) your run_spec, verbatim, with ONE extra term on the RHS
def run_spec_energy(panel, win, event_ids, signcol, vcov):
    sel = win[win["event_id"].isin(event_ids)]
    assert not sel[DATE].duplicated().any(), "duplicate treated date within a spec"
    treat_dates = set(sel[DATE])
    drop_dates  = set(win.loc[~win["event_id"].isin(event_ids), DATE]) - treat_dates
    df = panel[~panel[DATE].isin(drop_dates)].copy()
    df["sn"]  = df[DATE].map(sel.set_index(DATE)[signcol]).fillna(0)
    df["snx"] = df["sn"] * df[CARBON]
    df = df.dropna(subset=["AR", CARBON, "energy_x_carbon"])
    m = pf.feols(f"AR ~ {CARBON} + snx + energy_x_carbon | {ID} + {DATE}", data=df, vcov=vcov)
    return m, sel["event_id"].nunique(), len(df)

# 5) primary clean-risk spec, both windows -> the two Table 3 cells
TWOWAY = {"CRV1": f"{ID} + {DATE}"}
print("ENERGY x CARBON CONTROL  (clean-risk, two-way clustered)   snx in bp / 1-SD")
print("  reference (no energy term): announce +0.80 | pre -5.38")
for wlabel, w in [("ANNOUNCE [-1,+1]", announce), ("PRE-EVENT [-5,-2]", pre)]:
    m, nev, n = run_spec_energy(ar_e, w, clean_risk, "pv_ar_sign", TWOWAY)
    b, se = m.coef()["snx"]*1e4, m.se()["snx"]*1e4
    p     = m.pvalue()["snx"]
    lo, hi = [x*1e4 for x in m.confint().loc["snx"].tolist()]
    print(f"  {wlabel:18s} snx = {b:+.2f} bp (se {se:.2f}, p {p:.2f})  95% CI [{lo:+.2f}, {hi:+.2f}] | {nev} events, N={n:,}")

In [ ]:
m.pvalue()["snx"]